# 3. Evidence-grounded summaries and sentiment

**Model:** Qwen3.5-4B (GGUF, Q4_K_M) served locally by llama.cpp. No paid API. Temperature 0, fixed seed.

**Guardrails**
- Transcripts are delimited and treated as untrusted data.
- The JSON schema restricts citations to turn IDs in the call.
- Quotes are checked against the cited turn, and failures are retried at most twice.
- Citation times are copied from the transcript, never from the model.

**Sentiment**
- From transcript text alone. No vocal emotion is claimed.
- The windows are the first and last 3 substantive customer turns (at least 4 words, no backchannels, no courtesy-only turns).
- Each window is labelled in its own request, so one label cannot anchor the other.
- The change is computed by a rule from the two labels.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Image, Markdown, display

from calltrace.config import PROJECT_ROOT, load_config
from calltrace.data import AppTekDataset

pd.set_option("display.max_colwidth", 160, "display.max_rows", 80)
config = load_config()
dataset = AppTekDataset(config)
results = config.path("results")
evaluation_dir = results / "evaluation"


## Run analysis
This loads finished results. Missing calls need the server from `scripts/start_llm_server.py`.

In [2]:
from calltrace.export import export_all, load_processed_calls
from calltrace.pipeline import CallPipeline

records = dataset.load_manifest()["selected"]
print(CallPipeline(config, results).run(records))
export_all(config)
processed = load_processed_calls(results / "calls")

[001] already complete, skipped
[002] already complete, skipped
[003] already complete, skipped
[004] already complete, skipped
[005] already complete, skipped
[006] already complete, skipped
[007] already complete, skipped
[008] already complete, skipped
[009] already complete, skipped
[010] already complete, skipped
[011] already complete, skipped
[012] already complete, skipped
[013] already complete, skipped
[014] already complete, skipped
{'processed': [], 'failed': [], 'resumed': ['001', '002', '003', '004', '005', '006', '007', '008', '009', '010', '011', '012', '013', '014']}
Exported 14 calls and 839 turns.


## One call, followed from claim to evidence

In [3]:
transcript, analysis = processed[0]
turns = {t.turn_id: t for t in transcript.turns}
for name, claim in (("Issue", analysis.summary.customer_issue), ("Action", analysis.summary.agent_action),
                    ("Outcome", analysis.summary.outcome_or_next_step)):
    print(f"\n{name} ({claim.status}): {claim.text}")
    for item in claim.evidence:
        turn = turns[item.turn_id]
        print(f"   {item.turn_id} [{turn.start:.1f}-{turn.end:.1f}s] {turn.role}: \"{item.quote}\"")


Issue (stated): The customer accidentally transferred money to the wrong person via the banking app.
   001-T0002 [18.4-37.3s] Customer: "I've just been on the banking app and I've made a mess of things to be honest. I went onto the app, I've opened it up like normal. I had to make a transaction and when I've gone to transfer something, I've realized I've sent it to the wrong place."
   001-T0003 [37.3-50.6s] Customer: "So I've sent all this money and I'm really worried because obviously I didn't want to go to the wrong place and yeah. I don't know what to do because it's quite a bit of money and I really need it to go to where it's meant to."

Action (stated): The agent verified customer details, located the transaction, and guided the customer through the recall process.
   001-T0004 [51.5-70.9s] Agent: "Okay, I can understand how frustrating that's been. We've all been there. Just another manic Monday, as they say. just so I can bring up your details, could you please confirm the l

In [4]:
sentiment = analysis.customer_sentiment
print("Overall:", sentiment.overall.label, "-", sentiment.overall.justification)
print("Beginning:", sentiment.beginning.label, "| window:", sentiment.beginning.window_turn_ids)
print("Ending:", sentiment.ending.label, "| window:", sentiment.ending.window_turn_ids)
print("Change:", sentiment.change, "-", sentiment.change_note)

Overall: positive - The customer expresses relief and gratitude after successfully recalling the payment and receiving a summary for their real estate agent.
Beginning: negative | window: ['001-T0002', '001-T0003', '001-T0005']
Ending: positive | window: ['001-T0050', '001-T0053', '001-T0056']
Change: improved - Window labels moved from negative to positive.


## All calls

In [5]:
pd.read_csv(results / "summaries.csv")[["call_id", "customer_issue", "agent_action", "outcome_or_next_step", "outcome_status"]]

,call_id,customer_issue,agent_action,outcome_or_next_step,outcome_status
0,1,The customer accidentally transferred money to the wrong person via the banking app.,"The agent verified customer details, located the transaction, and guided the customer through the recall process.",The agent confirmed the recall was initiated and the refund would appear as pending within 24 to 48 hours.,stated
1,2,"The customer wanted a flexible, fully furnished apartment for a two-week business trip with amenities like a washer and dryer.",The agent recommended a fully furnished bachelor apartment with included appliances and flexible terms.,"The agent suggested the customer review availability online before booking, then promised to email details.",stated
2,3,The customer needed to schedule a follow-up appointment and request a prescription review.,"The agent collected patient details, checked availability, and booked the appointment.","The appointment was confirmed for Tuesday at 11 AM, and the prescription refill process was initiated.",stated
3,4,"The customer wanted to inquire about cabin availability, meal options, and pricing for a family cruise package to Dubai.","The agent explained package tiers, provided prices, confirmed room counts, and discussed food options and dietary needs.","The agent confirmed the booking details, requested a deposit, and promised to send a reservation email immediately.",stated
4,5,The customer wants to change their flight from tomorrow to Wednesday after six o'clock.,The agent checked availability and found the last seat available for the 9:30 flight.,"The agent requested payment details which the customer declined, and the call ended without confirming the change.",unclear
5,6,The customer requested a new electricity service for a home in Norton Beach.,The agent collected the customer's address and explained installation charges and plan differences.,The agent promised to send a sign-up link via email and scheduled installation for Sunday.,stated
6,7,"The customer wanted to inquire about catering services for a birthday brunch with 15 people, including dietary restrictions for gluten-free guests.","The agent explained menu options, pricing, delivery times, and additional service packages available for the event.",The agent agreed to send an email confirmation and noted that payment would be collected on the day of the event.,stated
7,8,The customer reported receiving ads despite having a premium subscription.,The agent verified the customer's email and guided them through clearing the app cache.,"The customer cleared the cache and confirmed ads were gone, but the agent offered further premium features.",stated
8,9,The customer needs to arrange a delivery for a fragile package containing expensive glassware to San Diego.,"The agent collected package details, recipient info, and discussed insurance and shipping options.",The agent confirmed the order details and scheduled the pickup for the afternoon.,stated
9,10,The customer reported Wi-Fi setup issues and later requested to add football channels.,The agent guided the customer through a modem reset and later offered to send feedback about the installer.,"The customer confirmed the Wi-Fi works after the reset, and the agent agreed to send an email regarding the installer.",stated


In [6]:
pd.read_csv(results / "sentiment.csv")[["call_id", "overall", "beginning", "ending", "change", "agent_sentiment_overall", "justification"]]

,call_id,overall,beginning,ending,change,agent_sentiment_overall,justification
0,1,positive,negative,positive,improved,positive,The customer expresses relief and gratitude after successfully recalling the payment and receiving a summary for their real estate agent.
1,2,positive,neutral,positive,improved,positive,The customer expresses clear interest and satisfaction by stating they would like to book the apartment and thanking the agent.
2,3,positive,neutral,positive,improved,positive,"The customer expresses relief, gratitude, and satisfaction with the service provided."
3,4,positive,neutral,positive,improved,positive,The customer expresses gratitude and relief upon learning the cruise accommodates dietary restrictions.
4,5,positive,neutral,positive,improved,positive,The customer expresses gratitude and relief at the agent's helpfulness despite their initial confusion.
5,6,positive,neutral,neutral,unchanged,positive,"The customer expresses satisfaction with the service details and the agent's helpfulness, concluding with a polite thank you."
6,7,positive,neutral,positive,improved,positive,"The customer expresses clear satisfaction and gratitude throughout the call, specifically praising the menu options, pricing, and service flexibility."
7,8,positive,negative,positive,improved,positive,The customer expresses relief and gratitude after the agent resolves their ad issue and provides helpful tips.
8,9,positive,neutral,positive,improved,positive,"The customer expresses relief and satisfaction with the service, stating the arrangement 'puts my mind at easy' and repeatedly confirming the outcome is 'pe..."
9,10,mixed,neutral,positive,improved,neutral,"The customer expresses frustration over the installer and the cost of the football package, but concludes with gratitude for the agent's help."


## Observed call handling
Describes single calls only. It is **not** an employee performance score, and agents are not ranked.

In [7]:
pd.read_csv(results / "call_handling.csv")[["call_id", "agent_acknowledged_issue", "concrete_action_or_next_step", "resolution"]]

,call_id,agent_acknowledged_issue,concrete_action_or_next_step,resolution
0,1,yes,yes,unresolved
1,2,yes,yes,unresolved
2,3,yes,yes,confirmed
3,4,yes,yes,confirmed
4,5,yes,no,unresolved
5,6,yes,yes,unresolved
6,7,yes,yes,unresolved
7,8,yes,yes,confirmed
8,9,yes,yes,confirmed
9,10,yes,yes,confirmed


## Minimum output per call
The assessment's table: call_id, speakers, transcript, summary, sentiment, and notes. Full text is in `results/calls.csv` and `results/calls.jsonl`.

In [8]:
table = pd.read_csv(results / "calls.csv")
table.assign(transcript=table.transcript.str.slice(0, 80) + " ...")[["call_id", "speakers", "transcript", "summary", "sentiment", "notes_limitations"]]

,call_id,speakers,transcript,summary,sentiment,notes_limitations
0,1,Speaker 1=Agent / Speaker 2=Customer,"[001-T0001 13.3-18.0s] Agent: Hi, thanks for calling A and Z. You are speaking w ...","Issue: The customer accidentally transferred money to the wrong person via the banking app. Action: The agent verified customer details, located the transac...",positive: The customer expresses relief and gratitude after successfully recalling the payment and receiving a summary for their real estate agent.,2% of turns flagged as low ASR confidence (uncalibrated heuristic). Sentiment comes from transcript text of a role-played call; it may not reflect real feel...
1,2,Speaker 1=Agent / Speaker 2=Customer,"[002-T0001 12.9-20.7s] Agent: Hello, thank you for calling the luxury apartment ...","Issue: The customer wanted a flexible, fully furnished apartment for a two-week business trip with amenities like a washer and dryer. Action: The agent reco...",positive: The customer expresses clear interest and satisfaction by stating they would like to book the apartment and thanking the agent.,Sentiment comes from transcript text of a role-played call; it may not reflect real feelings.
2,3,Speaker 1=Agent / Speaker 2=Customer,"[003-T0001 4.9-5.6s] Customer: Mm.\n[003-T0002 13.8-18.9s] Agent: Hi there, thank ...","Issue: The customer needed to schedule a follow-up appointment and request a prescription review. Action: The agent collected patient details, checked avail...","positive: The customer expresses relief, gratitude, and satisfaction with the service provided.",Sentiment comes from transcript text of a role-played call; it may not reflect real feelings.
3,4,Speaker 1=Agent / Speaker 2=Customer,"[004-T0001 13.5-22.1s] Agent: Good good morning. How may I help you today? Um, y ...","Issue: The customer wanted to inquire about cabin availability, meal options, and pricing for a family cruise package to Dubai. Action: The agent explained ...",positive: The customer expresses gratitude and relief upon learning the cruise accommodates dietary restrictions.,"1% of turns flagged as low ASR confidence (uncalibrated heuristic). Speakers overlap on 45% of the quieter speaker's time: the channels may be out of sync, ..."
4,5,Speaker 1=Agent / Speaker 2=Customer,"[005-T0001 1.5-7.5s] Customer: And the last four digits are five, zero, four, ni ...",Issue: The customer wants to change their flight from tomorrow to Wednesday after six o'clock. Action: The agent checked availability and found the last sea...,positive: The customer expresses gratitude and relief at the agent's helpfulness despite their initial confusion.,"Speakers overlap on 61% of the quieter speaker's time: the channels may be out of sync, so turn order, the summary, and the sentiment windows are unreliable..."
5,6,Speaker 1=Customer / Speaker 2=Agent,"[006-T0001 11.4-16.3s] Agent: Hello, this is Raccoon Energy. How may I help you? ...",Issue: The customer requested a new electricity service for a home in Norton Beach. Action: The agent collected the customer's address and explained install...,"positive: The customer expresses satisfaction with the service details and the agent's helpfulness, concluding with a polite thank you.",2% of turns flagged as low ASR confidence (uncalibrated heuristic). Sentiment comes from transcript text of a role-played call; it may not reflect real feel...
6,7,Speaker 1=Agent / Speaker 2=Customer,"[007-T0001 14.2-29.7s] Agent: Scenario seven, food and drink, customer plays as ...","Issue: The customer wanted to inquire about catering services for a birthday brunch with 15 people, including dietary restrictions for gluten-free guests. A...","positive: The customer expresses clear satisfaction and gratitude throughout the call, specifically praising the menu options, pricing, and service flexibil...",Sentiment comes from transcript text of a role-played call; it may not reflect real feelings.
7,8,Speaker 1=Agent / Speaker 2=Customer,"[008-T0001 3.8-10.1s] Agent: H